## Cell 1: Check GPU


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU')

## Cell 2: Install dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn accelerate

## Cell 3: Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Path to the folder you created in Google Drive
DATA_DIR = '/content/drive/MyDrive/contract-intelligence'
OUTPUT_DIR = '/content/drive/MyDrive/contract-intelligence/model_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Confirm the data files are there
for fname in ['train.jsonl', 'validation.jsonl', 'test.jsonl', 'categories.json']:
    path = os.path.join(DATA_DIR, fname)
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f'  {fname}: {"OK" if exists else "MISSING"} ({size:,} bytes)')

print('\nIf any file says MISSING, check the folder name in Google Drive matches: contract-intelligence')

## Cell 4: Load the dataset

In [ ]:
import json
from datasets import load_dataset

# Load categories
with open(os.path.join(DATA_DIR, 'categories.json')) as f:
    categories = json.load(f)
num_labels = len(categories)
print(f'Number of clause categories: {num_labels}')
print(f'First 5: {categories[:5]}')

# Load the JSONL splits
data_files = {
    'train':      os.path.join(DATA_DIR, 'train.jsonl'),
    'validation': os.path.join(DATA_DIR, 'validation.jsonl'),
    'test':       os.path.join(DATA_DIR, 'test.jsonl'),
}
raw_ds = load_dataset('json', data_files=data_files)
print('\nDataset loaded:')
print(raw_ds)

## Cell 5: Tokenize the text

Converts raw text into numbers the model can read.
`nlpaueb/legal-bert-base-uncased` is a BERT model pre-trained specifically
on legal documents — it understands legal language better than general BERT.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH = 256  # max tokens per chunk (256 is safe for 15GB T4 VRAM)

print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    enc = tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)
    enc['labels'] = [list(map(float, l)) for l in batch['labels']]
    return enc

print('Tokenizing dataset (this takes a few minutes)...')
tokenized_ds = raw_ds.map(
    tokenize_fn,
    batched=True,
    remove_columns=['text', 'contract_id']
)
print('Done.')
print(tokenized_ds)

## Cell 6: Load the model

In [ ]:
from transformers import AutoModelForSequenceClassification

print(f'Loading model: {MODEL_NAME} ...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type='multi_label_classification',
)
print('Model loaded.')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')

## Cell 7: Define evaluation metrics

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

THRESHOLD = 0.5  # probability above this = positive label

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)
    return {
        'f1_micro':        f1_score(labels, preds, average='micro',  zero_division=0),
        'f1_macro':        f1_score(labels, preds, average='macro',  zero_division=0),
        'precision_micro': precision_score(labels, preds, average='micro', zero_division=0),
        'recall_micro':    recall_score(labels, preds, average='micro',    zero_division=0),
    }

print('Metrics function ready.')

## Cell 8: Configure and run training



In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Try new arg name first, fall back to old name for older transformers versions
try:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        eval_strategy='epoch',
        save_strategy='epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1_micro',
        logging_steps=100,
        fp16=True,
        report_to='none',
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model='f1_micro',
        logging_steps=100,
        fp16=True,
        report_to='none',
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


trainer.train()

## Cell 9: Evaluate on the held-out test set



In [ ]:
print('Evaluating on test set...')
test_metrics = trainer.evaluate(tokenized_ds['test'])

print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f'  {k:<30}: {v:.4f}')

# Save metrics to Drive
metrics_path = os.path.join(OUTPUT_DIR, 'test_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(test_metrics, f, indent=2)
print(f'\nMetrics saved to: {metrics_path}')

## Cell 10: Save the final model to Google Drive

In [ ]:
final_model_dir = os.path.join(OUTPUT_DIR, 'final_model')
os.makedirs(final_model_dir, exist_ok=True)

trainer.save_model(final_model_dir)
tokenizer.save_pretrained(final_model_dir)


import shutil
shutil.copy(
    os.path.join(DATA_DIR, 'categories.json'),
    os.path.join(final_model_dir, 'categories.json')
)

print('Model saved to Google Drive at:')
print(f'  {final_model_dir}')
print('\nFiles saved:')
for f in sorted(os.listdir(final_model_dir)):
    size = os.path.getsize(os.path.join(final_model_dir, f))
    print(f'  {f:<40} {size:>10,} bytes')

print('\nDay 4-5 complete: fine-tuned clause classification model saved.')